# paired_v1 — explore

**Working directory:** Prefer **`experiments/04-audio-to-pro800-patch`** as cwd; the first code cell also works if the kernel cwd is **`dataset/paired_v1/`** (e.g. `nbconvert` / some IDE defaults).

**Kernel:** `gpu-audio-lab` (or this repo’s `.venv`).

**Dependencies:** `numpy`, `matplotlib`, `librosa`, `soundfile`, `IPython`. Optional **Starsky UMAP** plot loads `starsky/_index/meta.json` + `coords_knob_umap.npy` (built via `python -m starsky.build_index` after corpus deps and `.syx` under `starsky_bank/`).

This notebook:
1. Loads **`manifest.csv`** and decodes **`patch.json`** knob counts.
2. If the Starsky index exists: **knob-space UMAP** scatter with **paired_v1** pairs highlighted (join on `Path(bank_stem).stem`).
3. **Log-mel** spectrograms from **`render.wav`** (3 pairs, then 2×5 grid).
4. Optional **`IPython.display.Audio`** for two pairs.

**Verify index (PowerShell, repo root):**  
`Test-Path "experiments/04-audio-to-pro800-patch/starsky/_index/meta.json"`

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

# Support cwd = experiment 04 root OR cwd = this notebook folder (paired_v1).
_cwd = Path.cwd()
_manifest_here = _cwd / "manifest.csv"
_manifest_under = _cwd / "dataset" / "paired_v1" / "manifest.csv"
if _manifest_under.is_file():
    EXP_ROOT = _cwd.resolve()
    root = (_cwd / "dataset" / "paired_v1").resolve()
elif _manifest_here.is_file():
    root = _cwd.resolve()
    EXP_ROOT = root.parent.parent
else:
    raise AssertionError(
        "manifest.csv not found. Run with cwd = experiments/04-audio-to-pro800-patch "
        "or open the notebook from dataset/paired_v1/."
    )

rows: list[dict[str, str]] = []
with (root / "manifest.csv").open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        row["stem"] = Path(row["bank_stem"]).stem
        rows.append(row)

print(f"manifest: {len(rows)} rows")
for r in rows:
    print(r["pair_id"], r["stem"])

for i in range(1, 11):
    pj = root / f"pair_{i:02d}" / "patch.json"
    if pj.is_file():
        d = json.loads(pj.read_text(encoding="utf-8"))
        print(f"pair_{i:02d}", d.get("name"), "params", len(d.get("params_0_127", {})))

## Starsky knob-space UMAP (optional)

Join `manifest` **`stem`** to Starsky `meta[i]["stem"]`. If several index rows share a stem, prefer **`packet_ordinal == 0`**.

In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

idx_dir = EXP_ROOT / "starsky" / "_index"
meta_path = idx_dir / "meta.json"
umap_path = idx_dir / "coords_knob_umap.npy"

if not meta_path.is_file() or not umap_path.is_file():
    print("Starsky index not found. Build when ready:")
    print("  pip install -r requirements-corpus.txt")
    print("  pip install umap-learn   # optional; build_index falls back if missing")
    print("  # Place .syx under starsky_bank/ (or set STARSKY_PRESET_DIR)")
    print("  cd experiments/04-audio-to-pro800-patch")
    print("  python -m starsky.build_index")
else:
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    coords = np.load(umap_path)
    assert len(meta) == len(coords), (len(meta), coords.shape)

    by_stem: dict[str, list[int]] = defaultdict(list)
    for i, m in enumerate(meta):
        by_stem[m["stem"]].append(i)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(coords[:, 0], coords[:, 1], c="#bbbbbb", s=6, alpha=0.35, linewidths=0)

    cmap = plt.cm.tab10(np.linspace(0, 1, max(10, len(rows))))
    for j, row in enumerate(rows):
        stem = row["stem"]
        candidates = by_stem.get(stem, [])
        if not candidates:
            print("WARN: no Starsky row for stem", repr(stem), row["pair_id"])
            continue
        chosen = None
        for i in candidates:
            if meta[i].get("packet_ordinal", 0) == 0:
                chosen = i
                break
        if chosen is None:
            chosen = candidates[0]
        x, y = coords[chosen]
        ax.scatter(x, y, c=[cmap[j % 10]], s=70, edgecolors="black", linewidths=0.8, zorder=5)
        ax.annotate(
            row["pair_id"],
            (x, y),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
            fontweight="bold",
        )

    ax.set_title("Knob-space UMAP (Starsky), paired_v1 highlighted")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    legend_elements = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="#bbbbbb",
            markersize=6,
            label="Starsky bank",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="tab:blue",
            markeredgecolor="k",
            markersize=9,
            label="paired_v1",
        ),
    ]
    ax.legend(handles=legend_elements, loc="upper right")
    plt.tight_layout()
    plt.show()

## Log-mel spectrograms (`render.wav`)

Quick view of the first **three** manifest rows, then a **2×5** grid for all pairs.

In [ ]:
import librosa
import librosa.display

preview = rows[:3]
fig, axes = plt.subplots(len(preview), 1, figsize=(10, 2.8 * len(preview)))
if len(preview) == 1:
    axes = [axes]
for ax, row in zip(axes, preview):
    pid = row["pair_id"]
    wav = root / pid / "render.wav"
    y, sr = librosa.load(str(wav), sr=None, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(f"{pid} — {row['stem']}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 5.5))
for ax, row in zip(axes.flat, rows):
    pid = row["pair_id"]
    wav = root / pid / "render.wav"
    y, sr = librosa.load(str(wav), sr=None, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(f"{pid}\n{row['stem']}", fontsize=8)
plt.suptitle("All pairs — log-mel", y=1.02)
plt.tight_layout()
plt.show()

## What this tests

**Log-mel** gives a fast **visual** check that each `render.wav` is a real capture (energy, pitch band, decay), not silence or a bad export. **Knob-space UMAP** (when `starsky/_index` exists) shows where each paired preset sits relative to the **indexed bank**—useful sanity before any audio→patch or retrieval experiments. Together they anchor **heard audio** to **decoded patch / library geometry**.

In [ ]:
from IPython.display import Audio, display

for pid in ("pair_01", "pair_02"):
    wav = root / pid / "render.wav"
    y, sr = librosa.load(str(wav), sr=None, mono=True)
    print(pid, "sr=", sr, "samples=", len(y))
    display(Audio(y, rate=sr))